# Manual Implementation of Image Stitching

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

In [ ]:
# Path of the folder containing the images that will be used for stitching
IMAGE_FOLDER = "/content/images"

# List of allowed image file extensions that the program will load
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

# Function to resize images before stitching
# This helps reduce memory usage and speeds up feature detection/matching
def resize_for_stitching(img, max_dim=800):

    # Get the height (h) and width (w) of the input image
    h, w = img.shape[:2]

    # Calculate scaling factor based on the largest dimension (height or width)
    # The goal is to make the largest side equal to max_dim
    scale = max_dim / max(h, w)

    # Only resize if the image is larger than the allowed maximum dimension
    # (i.e., scale < 1 means the image needs to be reduced)
    if scale < 1:
        # Resize the image while maintaining the original aspect ratio
        img = cv2.resize(img, (int(w * scale), int(h * scale)))

    # Return the resized image (or original if resizing was not needed)
    return img

In [ ]:
# Function to load images from a folder
def load_images(folder):

    # List to store all loaded images
    images = []

    # Get all file names in the folder and sort them
    # Sorting ensures images are processed in order (important for stitching)
    files = sorted(os.listdir(folder))

    # Loop through every file in the folder
    for file in files:

        # Get the file extension (e.g., .jpg, .png)
        # Convert to lowercase to avoid case issues (.JPG vs .jpg)
        ext = Path(file).suffix.lower()

        # Check if the file extension matches allowed image formats
        if ext in IMAGE_EXTENSIONS:

            # Create the full file path
            path = os.path.join(folder, file)

            # Read the image using OpenCV
            img = cv2.imread(path)

            # Resize the image to reduce computation for stitching
            img = resize_for_stitching(img)

            # Check if the image was successfully loaded
            if img is not None:
                # Add the image to the images list
                images.append(img)

                # Print confirmation with image name and dimensions
                print(f"Loaded: {file}  shape={img.shape}")

    # Print total number of images successfully loaded
    print(f"\nTotal images loaded: {len(images)}")

    # Return the list of loaded images
    return images

In [ ]:
def show_images(images, titles=None, figsize=(15,6)):

    n = len(images)

    plt.figure(figsize=figsize)

    for i, img in enumerate(images):

        plt.subplot(1, n, i+1)

        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if titles:
            plt.title(titles[i])

        else:
            plt.title(f"Image {i+1}")

        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Function to detect keypoints and match features between two images
def detect_and_match_features(img1, img2):

    # Convert both images to grayscale
    # Feature detectors usually work better on grayscale images
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # Create an ORB (Oriented FAST and Rotated BRIEF) feature detector
    # nfeatures=2000 means ORB will try to detect up to 2000 keypoints
    orb = cv2.ORB_create(nfeatures=2000)

    # Detect keypoints and compute descriptors for both images
    # kp = keypoints (important points in the image)
    # des = descriptors (numerical representation of those keypoints)
    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)

    # Create a Brute Force matcher using Hamming distance
    # Hamming distance works well with ORB descriptors (binary descriptors)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING)

    # Find the 2 nearest matches for each descriptor
    # k=2 is used for applying Lowe's Ratio Test
    matches = bf.knnMatch(des1, des2, k=2)

    # List to store good matches after filtering
    good_matches = []

    # Apply Lowe's Ratio Test to remove ambiguous matches
    for m, n in matches:

        # If the best match is significantly better than the second-best
        # we accept it as a reliable match
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    # Print the number of reliable matches found
    print("Good matches:", len(good_matches))

    # Return keypoints, descriptors, and the filtered matches
    return kp1, des1, kp2, des2, good_matches

In [ ]:
# Function to compute the homography matrix between two images
def compute_homography(kp1, kp2, matches, min_matches=10):

    # Check if the number of matches is sufficient to estimate homography
    # Homography requires at least 4 matches, but we usually use a higher threshold
    if len(matches) < min_matches:
        return None, None

    # Extract the coordinates of matched keypoints from image 1
    # queryIdx refers to the index of the keypoint in the first image
    src_pts = np.float32(
        [kp1[m.queryIdx].pt for m in matches]
    ).reshape(-1,1,2)

    # Extract the coordinates of matched keypoints from image 2
    # trainIdx refers to the index of the keypoint in the second image
    dst_pts = np.float32(
        [kp2[m.trainIdx].pt for m in matches]
    ).reshape(-1,1,2)

    # Compute the homography matrix using RANSAC
    # Homography describes the geometric transformation between two images
    # RANSAC helps remove incorrect matches (outliers)
    H, mask = cv2.findHomography(
        src_pts,      # source points (image 1)
        dst_pts,      # destination points (image 2)
        cv2.RANSAC,   # robust method to filter incorrect matches
        5.0           # reprojection threshold
    )

    # H = 3x3 homography matrix used to warp one image to another
    # mask = array indicating which matches are inliers (good matches)

    return H, mask

In [ ]:
# Function to stitch two images using a homography matrix
def stitch_two_images(img1, img2, H):

    # Get height and width of both images
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # Define the corner points of the second image
    # (top-left, bottom-left, bottom-right, top-right)
    corners = np.float32(
        [[0,0],[0,h2],[w2,h2],[w2,0]]
    ).reshape(-1,1,2)

    # Transform these corners using the homography matrix
    # This shows where img2 will appear after warping
    warped_corners = cv2.perspectiveTransform(corners, H)

    # Combine corners of image1 and warped corners of image2
    # This helps determine the size of the final panorama canvas
    all_corners = np.concatenate(
        (
            np.float32([[0,0],[0,h1],[w1,h1],[w1,0]]).reshape(-1,1,2),
            warped_corners
        ),
        axis=0
    )

    # Find minimum and maximum coordinates for the panorama canvas
    xmin, ymin = np.int32(all_corners.min(axis=0).ravel() - 0.5)
    xmax, ymax = np.int32(all_corners.max(axis=0).ravel() + 0.5)

    # Create a translation matrix
    # This shifts the stitched result so that all coordinates are positive
    translation = np.array(
        [[1,0,-xmin],
         [0,1,-ymin],
         [0,0,1]]
    )

    # Warp the second image using the homography + translation
    # warpPerspective applies the geometric transformation
    result = cv2.warpPerspective(
        img2,
        translation.dot(H),
        (xmax-xmin, ymax-ymin)
    )

    # Place the first image onto the panorama canvas
    # This overlays img1 in the correct position
    result[-ymin:h1-ymin, -xmin:w1-xmin] = img1

    # Return the stitched panorama
    return result

In [ ]:
# Function to stitch multiple images sequentially into a single panorama
def simple_stitch_all(images):

    # Start the panorama with the first image
    # copy() prevents modifying the original image
    panorama = images[0].copy()

    # Loop through the remaining images one by one
    for i in range(1, len(images)):

        # Print which image is currently being stitched
        print(f"\nStitching image {i}")

        # Detect keypoints and match features between
        # the current panorama and the next image
        kp1, des1, kp2, des2, matches = detect_and_match_features(
            panorama,
            images[i]
        )

        # Check if there are enough good matches
        # Without enough matches, homography cannot be computed reliably
        if len(matches) < 10:
            print("Not enough matches")
            continue

        # Compute the homography matrix between panorama and new image
        H, mask = compute_homography(kp1, kp2, matches)

        # If homography computation fails, skip this image
        if H is None:
            print("Homography failed")
            continue

        # Warp the new image and merge it with the existing panorama
        panorama = stitch_two_images(
            panorama,
            images[i],
            H
        )

    # Return the final stitched panorama
    return panorama

In [ ]:
images = load_images(IMAGE_FOLDER)

if len(images) < 2:
    raise ValueError("Need at least 2 images")

show_images(
    images[:4],
    [f"Input {i+1}" for i in range(min(4,len(images)))]
)

print("\nRunning manual stitching...\n")

panorama = simple_stitch_all(images)

In [ ]:
plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
plt.axis("off")

# Using OpenCV Function

In [ ]:
import cv2

In [ ]:
import numpy as np
import cv2
import os

# Path to folder containing images
image_folder = "/content/images"

# Output stitched image path
output_path = "/content/stitched_output.jpg"

In [ ]:
# Print a message indicating that the image loading process is starting
print("[INFO] loading images...")

# Import required libraries
import os          # Used for file and directory operations
import cv2         # OpenCV library for image processing

# Path to the folder containing the images
image_folder = "/content/images"

# List to store loaded images
images = []

# Get all file names in the folder and sort them
# Sorting ensures images are processed in a consistent order
imagePaths = sorted(os.listdir(image_folder))

# Loop through each file name in the folder
for imagePath in imagePaths:

    # Create the full path of the image file
    full_path = os.path.join(image_folder, imagePath)

    # Read the image using OpenCV
    image = cv2.imread(full_path)

    # Check if the image was loaded successfully
    if image is not None:
        # Add the image to the images list
        images.append(image)

[INFO] loading images...


In [ ]:
# Create an OpenCV Stitcher object
# This object contains the full panorama stitching pipeline
stitcher = cv2.Stitcher_create()

# Perform the stitching process on the list of images
# status -> indicates whether stitching succeeded or failed
# stitched -> the final stitched panorama image
status, stitched = stitcher.stitch(images)

In [ ]:
# if stitching succeeded
if status == 0:

    # enable or disable cropping
    crop = True

    if crop:
        print("[INFO] cropping...")

        # add small border to stitched image
        stitched = cv2.copyMakeBorder(
            stitched, 10, 10, 10, 10,
            cv2.BORDER_CONSTANT, (0, 0, 0)
        )

        # convert to grayscale
        gray = cv2.cvtColor(stitched, cv2.COLOR_BGR2GRAY)

        # threshold image
        thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY)[1]

[INFO] cropping...


In [ ]:
# find external contours
cnts, _ = cv2.findContours(
    thresh.copy(),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# get the largest contour (stitched panorama boundary)
c = max(cnts, key=cv2.contourArea)

# create mask for bounding rectangle
mask = np.zeros(thresh.shape, dtype="uint8")

(x, y, w, h) = cv2.boundingRect(c)

# draw bounding rectangle on mask
cv2.rectangle(mask, (x, y), (x + w, y + h), 255, -1)

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)

In [ ]:
# create two copies of the mask
minRect = mask.copy()
sub = mask.copy()

# keep looping until there are no non-zero pixels left
while cv2.countNonZero(sub) > 0:

    # erode the rectangle mask
    minRect = cv2.erode(minRect, None)

    # subtract threshold image from eroded mask
    sub = cv2.subtract(minRect, thresh)

In [ ]:
# find contours in the minimum rectangular mask
cnts, _ = cv2.findContours(
    minRect.copy(),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# get largest contour
c = max(cnts, key=cv2.contourArea)

# compute bounding box
(x, y, w, h) = cv2.boundingRect(c)

# crop the stitched panorama
stitched = stitched[y:y + h, x:x + w]

In [ ]:
from google.colab.patches import cv2_imshow

# if stitching succeeded
if status == 0:

    # save stitched image
    cv2.imwrite(output_path, stitched)

    # display stitched panorama
    cv2_imshow(stitched)

# otherwise stitching failed
else:
    print("[INFO] image stitching failed ({})".format(status))